# DiDAQt — Butterfly demonstration on FABRIC (BMv2)

This notebook builds the DiDAQt failover testbed on the **butterfly topology**
from the paper, using **real FABRIC nodes for every node of the butterfly** and
**BMv2** (`simple_switch`, via fablib's `Attestable_Switch`) as the software P4
data plane.

## Two sites, two slices

At full scale the single-slice version of this topology (~148 nodes once every
butterfly link becomes a monitored L2Bridge — each monitored link adds a
monitor node and two sub-networks) is too large for one FABRIC slice-submit and
returns **HTTP 413 (payload too large)**. This notebook instead splits the
butterfly **horizontally by the top bit of every index**: switches/senders/
receivers with index bit `K-1` clear go to site **RUTG** (slice `-a`), the rest
go to site **PRIN** (slice `-b`). Each half is built and submitted as its own
`CrinkleSlice` (its own analyzer, its own monitors), which keeps each
submission roughly half size.

Splitting by the *top* index bit means the **only** switch-to-switch links that
cross the site boundary are the `out_cross`/`in_cross` edges between the
**last two** switch ranks (rank `K-1` -> rank `K`) — see the config/math cell
for why. Two independent FABRIC slices cannot share an L2Bridge/L2STS network,
so these `WIDTH` directed cross-site edges (`WIDTH/2` RUTG->PRIN, `WIDTH/2`
PRIN->RUTG) are carried over **FABNet L3** instead: a small per-slice **tunnel
node** VXLAN-encapsulates the raw butterfly frames and routes them to its peer
tunnel node in the other slice, which decapsulates back onto an ordinary local
L2 segment into the destination switch. Crinkle's monitor only ever sees plain
local L2 traffic — the DPDK monitor itself is unmodified. Making Crinkle aware
of this *local* L2 segment of a cross-slice link is the fork's new
`CrinkleSlice.add_monitored_l3network()` (see the "install the fork" cell
below).

## Topology (defaults)

```
 16 senders            4 switch ranks x 8 = 32 BMv2 switches          8 receivers
 (8 pairs)         rank0      rank1      rank2      rank3
   snd1 \                                          site RUTG | site PRIN
   snd2  >--- sw0n0 --- sw1n0 --- sw2n0 --- sw3n0 --------------------- rcv0
   snd3 \        \  X   /   \  X  /   \  = tunnel (FABNet/VXLAN) =
   snd4  >--- sw0n1 --- sw1n1 --- sw2n1 --- sw3n1 --------------------- rcv1
    ...    (stages 0..K-2: ordinary intra-site cross-links, stage s flips     ...
   snd15\   bit s of the destination index; stage K-1 is the ONLY stage       ...
   snd16 >--- sw0n7 --- sw1n7 --- sw2n7 --- sw3n7 --------------------- rcv7
```

* **Butterfly self-routing**: a frame's destination MAC is always a receiver's
  NIC MAC. At rank *s*, switch *i* forwards toward its *straight* next hop if
  bit `s` of the destination receiver index equals bit `s` of *i*, else toward
  its *cross* next hop (`i XOR 2^s`). After all `K` stages the frame arrives at
  rank-`K` switch = receiver index. This gives every sender a path to **any**
  receiver — the redundancy DiDAQt exploits for failover (future work), and
  also exactly what the cross-site probe below relies on: switches are
  programmed with forwarding rules for **every** receiver, not just the
  initial mapping, so no BMv2 rule change is needed to route a frame
  cross-site — pointing a sender at a different destination MAC is enough.
* Each **intra-site** butterfly link is created with **`add_monitored_l2network`**
  (single-site **L2Bridge**); each **cross-site** butterfly link (stage `K-1`
  cross edges) is created with **`add_monitored_l3network`** (the local half
  of an L2Bridge, plus tunnel metadata) so a Crinkle **monitor** node sits
  transparently on every link and reports to that slice's **analyzer**.
* Two **controller** nodes run in a main/follower HA pair (`controller_ha.py`):
  the main lives in the RUTG slice, the follower in the PRIN slice, and they
  coordinate over FABNet; the follower takes over if the main stops
  heart-beating.

## This notebook's goal

Create both slices and **verify reachability** two ways:

1. Every sender's workload reaches its assigned receiver through the
   butterfly (as before). **Important:** the initial mapping (sender pair `p`
   -> receiver `p`) is intra-site *by construction* (pair `p` and receiver `p`
   share the same index, hence the same half), so this check alone never
   exercises the new cross-site VXLAN tunnels.
2. A **cross-site probe**: a short burst where one RUTG sender's frames are
   pointed at a PRIN receiver, and vice versa, confirming the VXLAN tunnel
   actually carries butterfly frames end to end.

Introducing failures / measuring failover is later work.

> **Scale warning.** With the default full-spec dimensions this is still a big
> pair of slices: ~16 senders + 32 switches + 8 receivers + 2 controllers + 2
> tunnel nodes + 2 analyzers + ~72 per-link monitors, split roughly in half
> across two sites. Shrink `K` / `SENDERS_PER_SW` in the config cell for a
> smoke test first (`K=1` is a good starting point).


## 0. Install the Crinkle L3-monitoring fablib fork

The cross-site links need `CrinkleSlice.add_monitored_l3network()`, which does
not exist in upstream fablib yet — it lives on the `support-l3` branch of a
fork. Install it **before** importing `fabrictestbed_extensions` below.

In [ ]:
import subprocess
import sys

# Install the support-l3 fablib fork (adds Crinkle L3 monitoring:
# CrinkleSlice.add_monitored_l3network(), needed below for the cross-site
# FABNet links). Default: install straight from the branch on GitHub.
#
#   !pip install -q "git+https://github.com/awolosewicz/fabrictestbed-extensions.git@support-l3"
#
# When iterating locally against an editable checkout of the fork instead,
# comment the git-URL install below and uncomment the local -e install:
#
#   !pip install -q -e /path/to/fabrictestbed-extensions
#
# (Using subprocess rather than the `!pip` shell-magic so this cell is plain,
# compilable Python -- convenient for offline notebook linting.)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "git+https://github.com/awolosewicz/fabrictestbed-extensions.git@support-l3",
])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
#                        "-e", "/path/to/fabrictestbed-extensions"])
print("installed support-l3 fablib fork")

## 1. Fablib setup

In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager

fablib = FablibManager()
fablib.show_config()

## 2. Configuration & butterfly math

All knobs live here so the notebook can be resumed and rescaled from one place.

In [ ]:
import os
import json
from concurrent import futures

# ---- Placement: two sites, two slices (each ~half the butterfly) ----
SITE_A = "CERN"                   # top-bit-clear half; see fablib.list_sites()
SITE_B = "PRIN"                   # top-bit-set half
SLICE_NAME = "didaqt-butterfly"
SLICE_A = f"{SLICE_NAME}-a"
SLICE_B = f"{SLICE_NAME}-b"

# If a tunnel node's NIC count (WIDTH data NICs + 1 FABNet uplink) is too high
# for one node's VF budget, split each slice's tunnel into an egress node
# (tun{X}e, one NIC per outgoing cross-site link) and an ingress node
# (tun{X}i, one NIC per incoming cross-site link). Off by default: a single
# tunnel node per slice handling both directions.
TUNNEL_SPLIT = False

# ---- Butterfly dimensions ----
K              = 3                # butterfly dimension; WIDTH = 2**K
WIDTH          = 2 ** K          # switches per rank (8)
SWITCH_RANKS   = K + 1           # rank 0 .. K  (4 ranks, K=3 stages between them)
SENDERS_PER_SW = 1               # senders feeding each rank-0 switch (pairs of 2)

NUM_SENDERS   = WIDTH * SENDERS_PER_SW   # 16
NUM_RECEIVERS = WIDTH                     # 8
NUM_SWITCHES  = WIDTH * SWITCH_RANKS      # 32

# ---- Node sizing (kept small; BMv2 + sender/receiver are light) ----
SW_CORES,  SW_RAM,  SW_DISK  = 2, 4, 10   # per BMv2 switch
END_CORES, END_RAM, END_DISK = 2, 4, 10   # per sender / receiver / controller / tunnel node

# ---- Names / images / paths ----
END_IMAGE  = "default_ubuntu_22"          # senders/receivers/controllers/tunnels
NAME_PREFIX = "C"                          # Crinkle resource prefix (analyzer/monitors)

REPO       = os.path.abspath("..")         # repo root (this notebook lives in artifact/)
REMOTE_DIR = "/home/ubuntu/didaqt"         # upload target on end hosts
RUN_DIR    = "/tmp/didaqt_run"             # per-run logs on end hosts

P4_LOCAL   = os.path.join(REPO, "examples/p4/l2_forward_bmv2.p4")
P4_TABLE   = "MyIngress.l2_forward"        # table name as compiled (see show_tables)

# ---- DiDAQt runtime params ----
HB_PORT    = 9000                          # receiver -> controller heartbeat UDP port
HA_PORT    = 9100                          # controller <-> controller HA heartbeat port
SEND_RATE  = 500                           # frames/sec/sender (BMv2-friendly)
# The Crinkle monitor reports each packet to the analyzer as
# [88-byte header][frame + 16-byte UID trailer] over a default-MTU
# FABNet path (crease/monitor_source.c), so frames longer than 1410
# bytes forward fine but never reach the provenance store.
SEND_LEN   = 1400                          # frame bytes (sender -l)
TEST_SECS  = 20                            # workload duration
PROBE_SECS = 5                             # cross-site probe burst duration

# =====================================================================
# Butterfly helpers (pure functions of the config above)
# =====================================================================
def bit(x, b):
    return (x >> b) & 1

def cross_bit(stage):
    """The index bit flipped by the 'cross' edge at a given stage (0..K-1).

    Bit-consumption order is chosen so stage `s` fixes bit `s` (rather than
    bit K-1-s): stage K-1 is then the ONLY stage whose cross edge flips the
    MSB (bit K-1) -- i.e. the only stage whose cross edges cross the site
    boundary (see half()/site_of() below). Self-routing still delivers
    correctly for every (entry, receiver) pair: all K bits get fixed by the
    time a frame reaches rank K, regardless of the order they're fixed in.
    """
    return 1 << stage

def switch_ports(rank):
    """Ordered BMv2 port list for a switch at the given rank.
    BMv2 port index == position in this list (Attestable_Switch.start_switch)."""
    ins = [f"in_s{j}" for j in range(SENDERS_PER_SW)] if rank == 0 else ["in_str", "in_cross"]
    outs = ["out_str", "out_cross"] if rank < K else ["out_rx"]
    return ins + outs

def sw_name(rank, idx):
    return f"sw{rank}n{idx}"

def snd_name(sid):
    return f"snd{sid}"

def rcv_name(r):
    return f"rcv{r}"

# sender ids are 1..NUM_SENDERS; pair p owns ids [p*SPS+1 .. p*SPS+SPS],
# and every sender in pair p is initially routed to receiver p.
def pair_of_sender(sid):
    return (sid - 1) // SENDERS_PER_SW

def egress_index(rank, node_idx, recv_idx, ports):
    """Return the BMv2 egress port index for frames destined to receiver
    recv_idx at switch (rank, node_idx), or None to drop (default)."""
    if rank == K:
        return ports.index("out_rx") if recv_idx == node_idx else None
    b = rank
    straight = bit(node_idx, b) == bit(recv_idx, b)
    return ports.index("out_str" if straight else "out_cross")

# ---- Two-site split: horizontal, by the MSB (bit K-1) of an index ----
def half(idx):
    """0 -> site/slice A (RUTG), 1 -> site/slice B (PRIN)."""
    return bit(idx, K - 1)

def site_of(idx):
    return SITE_A if half(idx) == 0 else SITE_B

def slice_name_of(idx):
    return SLICE_A if half(idx) == 0 else SLICE_B

def slice_letter_of(idx):
    return "A" if half(idx) == 0 else "B"

# A switch (rank, idx) belongs to the half given by idx alone (every rank of a
# given column idx is in the same half). Senders live with their rank-0 entry
# switch: pair p -> rank-0 node p. Receiver r attaches to rank-K switch idx=r.

# ---- Cross-site directed links: stage K-1 CROSS edges only ----
# (K-1, i).out_cross -> (K, i ^ (1 << (K-1))).in_cross, for every i in 0..WIDTH-1.
# WIDTH directed edges total: WIDTH/2 A->B, WIDTH/2 B->A. Deterministic VNI per
# edge so both slices (built independently, see build cells) agree on naming.
def cross_link_vni(i):
    return 1000 + i

CROSS_LINKS = []   # one entry per stage-(K-1) cross edge, keyed by source index i
for i in range(WIDTH):
    j = i ^ cross_bit(K - 1)
    CROSS_LINKS.append({
        "i": i, "j": j, "vni": cross_link_vni(i),
        "src_switch": (K - 1, i), "dst_switch": (K, j),
        "src_letter": slice_letter_of(i), "dst_letter": slice_letter_of(j),
    })
assert all(l["src_letter"] != l["dst_letter"] for l in CROSS_LINKS), \
    "every stage K-1 cross edge must cross the site boundary"

def tunnel_node_name(letter, direction):
    """direction: 'out' (this slice is the cross-link source) or
    'in' (this slice is the cross-link destination)."""
    if TUNNEL_SPLIT:
        return f"tun{letter}{'e' if direction == 'out' else 'i'}"
    return f"tun{letter}"

# ---- Report ----
n_sender_links = NUM_SENDERS
n_intra_stage_links = (K - 1) * WIDTH * 2 + WIDTH * 2   # stages 0..K-2 (str+cross) + stage K-1 straight
n_cross_site_links  = len(CROSS_LINKS)
n_recv_links   = NUM_RECEIVERS
n_links = n_sender_links + n_intra_stage_links + n_cross_site_links + n_recv_links

print(f"Butterfly K={K}  WIDTH={WIDTH}  switch ranks={SWITCH_RANKS} (stages={K})")
print(f"  senders={NUM_SENDERS}  switches={NUM_SWITCHES}  receivers={NUM_RECEIVERS}")
print(f"  links: sender={n_sender_links} intra-site-stage={n_intra_stage_links} "
      f"cross-site={n_cross_site_links} receiver={n_recv_links}  total={n_links}")
print(f"  cross-site VNIs: {[l['vni'] for l in CROSS_LINKS]}")
print(f"  site A={SITE_A} (slice {SLICE_A})   site B={SITE_B} (slice {SLICE_B})")
print(f"  tunnel node NICs per slice: {WIDTH} data + 1 FABNet "
      f"({'split into e/i' if TUNNEL_SPLIT else 'single node'})")

# ---- Cross-site probe targets (see verify cell): pick one sender/receiver
# pair per direction whose halves differ, to actually exercise a tunnel. ----
PROBE_AB_SENDER   = 1                                   # pair 0 (site A) sender
PROBE_AB_RECEIVER = WIDTH - 1                            # site B receiver (MSB set)
PROBE_BA_SENDER   = (WIDTH - 1) * SENDERS_PER_SW + 1      # pair (WIDTH-1) (site B) sender
PROBE_BA_RECEIVER = 0                                    # site A receiver
assert slice_letter_of(pair_of_sender(PROBE_AB_SENDER)) != slice_letter_of(PROBE_AB_RECEIVER)
assert slice_letter_of(pair_of_sender(PROBE_BA_SENDER)) != slice_letter_of(PROBE_BA_RECEIVER)

## 3. Build the two Crinkle slices

Order matters within each slice: `add_analyzer` must be called **before** any
monitored network. `build_slice(letter)` builds one site's half: its analyzer,
its BMv2 switches / senders / receivers / controller, its per-site control
network, every **intra-site** butterfly link as a monitored L2Bridge
(`add_monitored_l2network`), and every **cross-site** butterfly link's local
half as a monitored L3 segment (`add_monitored_l3network`) terminating on that
slice's tunnel node. Crucially, each call only ever touches interfaces that
live in its own slice — `build_slice("A")` and `build_slice("B")` do not need
each other's live objects, they just agree on VNIs/naming via the pure-Python
`CROSS_LINKS` table above. The actual VXLAN tunnel is wired up in a separate
post-submit cell once both slices are active and FABNet IPs are known.

In [ ]:
def build_slice(letter):
    """Build one site's half of the butterfly (site RUTG=A, PRIN=B)."""
    site = SITE_A if letter == "A" else SITE_B
    slice_name = SLICE_A if letter == "A" else SLICE_B
    s = fablib.new_crinkle_slice(name=slice_name, name_prefix=NAME_PREFIX)

    # --- Analyzer first (required before monitored networks) ---
    s.add_analyzer(site=site)

    # --- BMv2 switches in this half: sw{rank}n{idx}, ports per switch_ports() ---
    switches = {}                      # (rank, idx) -> Attestable_Switch
    for rank in range(SWITCH_RANKS):
        for idx in range(WIDTH):
            if slice_letter_of(idx) != letter:
                continue
            sw = s.add_attestable_switch(
                name=sw_name(rank, idx), site=site, ports=switch_ports(rank),
                cores=SW_CORES, ram=SW_RAM, disk=SW_DISK,
            )
            switches[(rank, idx)] = sw

    # --- Senders in this half: one NIC_Basic 'd' (data) each ---
    senders = {}                       # sid -> node
    for sid in range(1, NUM_SENDERS + 1):
        p = pair_of_sender(sid)
        if slice_letter_of(p) != letter:
            continue
        n = s.add_node(name=snd_name(sid), site=site,
                       cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
        n.add_component(model="NIC_Basic", name="d")
        senders[sid] = n

    # --- Receivers in this half: 'd' (data) + 'c' (control) ---
    receivers = {}                     # r -> node
    for r in range(NUM_RECEIVERS):
        if slice_letter_of(r) != letter:
            continue
        n = s.add_node(name=rcv_name(r), site=site,
                       cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
        n.add_component(model="NIC_Basic", name="d")
        n.add_component(model="NIC_Basic", name="c")
        receivers[r] = n

    # --- One controller per slice: main in A, follower in B ---
    ctl_name = "ctlmain" if letter == "A" else "ctlfollow"
    ctl = s.add_node(name=ctl_name, site=site,
                     cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
    ctl.add_component(model="NIC_Basic", name="c")

    # --- Per-slice control network (FABNetv4) for heartbeats ---
    # Every control node also gets a route to the whole FABNet v4 supernet so
    # heartbeats/HA traffic can reach the *other* slice's controller/receivers
    # over FABNet (mirrors what Node.add_fabnet() does for a single node).
    ctrl_ifaces = [receivers[r].get_component(name="c").get_interfaces()[0] for r in receivers]
    ctrl_ifaces += [ctl.get_component(name="c").get_interfaces()[0]]
    for iface in ctrl_ifaces:
        iface.set_mode("auto")
    ctrl_net = s.add_l3network(name=f"ctlnet_{letter}", interfaces=ctrl_ifaces, type="IPv4")
    for iface in ctrl_ifaces:
        iface.get_node().add_route(subnet=fablib.FABNETV4_SUBNET, next_hop=ctrl_net.get_gateway())

    def data_iface(node):
        return node.get_component(name="d").get_interfaces()[0]

    def link(name, up_iface, down_iface):
        """Intra-site butterfly link: monitored L2Bridge (sink = downstream iface)."""
        up_iface.set_mode("manual")
        down_iface.set_mode("manual")
        s.add_monitored_l2network(
            name=name, interfaces=[up_iface, down_iface], sinks=[down_iface], site=site,
        )

    # 1) sender -> rank-0 switch (always intra-site: a sender lives with its
    #    rank-0 entry node by construction)
    for p in range(WIDTH):
        if slice_letter_of(p) != letter:
            continue
        ports = switch_ports(0)
        for j in range(SENDERS_PER_SW):
            sid = p * SENDERS_PER_SW + j + 1
            sw_if = switches[(0, p)].get_port_interface(ports[j])
            link(f"lsnd{sid}", data_iface(senders[sid]), sw_if)

    # 2) inter-rank butterfly stages: straight edges (all stages) and cross
    #    edges for stages 0..K-2 are always intra-site (cross_bit(s) < 2**(K-1)
    #    for s < K-1, so it never flips the half-determining MSB). Stage K-1's
    #    cross edges are the cross-site links, handled separately below.
    for stage in range(K):
        for i in range(WIDTH):
            if slice_letter_of(i) != letter:
                continue
            up = switches[(stage, i)]
            link(f"l{stage}str{i}",
                 up.get_port_interface("out_str"),
                 switches[(stage + 1, i)].get_port_interface("in_str"))
            if stage < K - 1:
                j = i ^ cross_bit(stage)
                link(f"l{stage}x{i}",
                     up.get_port_interface("out_cross"),
                     switches[(stage + 1, j)].get_port_interface("in_cross"))
            # stage == K-1: out_cross is a cross-site link, wired via the
            # tunnel node below instead of a plain add_monitored_l2network().

    # 3) rank-K switch -> receiver
    for r in range(NUM_RECEIVERS):
        if slice_letter_of(r) != letter:
            continue
        link(f"lrcv{r}",
             switches[(K, r)].get_port_interface("out_rx"),
             data_iface(receivers[r]))

    # --- Tunnel node(s): carry this slice's stage-(K-1) cross-site edges ---
    out_links = [l for l in CROSS_LINKS if l["src_letter"] == letter]   # this slice is the src
    in_links  = [l for l in CROSS_LINKS if l["dst_letter"] == letter]   # this slice is the dst

    def make_tunnel_node(nname):
        n = s.add_node(name=nname, site=site,
                       cores=END_CORES, ram=END_RAM, disk=END_DISK, image=END_IMAGE)
        fnet_iface = n.add_component(model="NIC_Basic", name="f").get_interfaces()[0]
        fnet_iface.set_mode("auto")
        fnet = s.add_l3network(name=f"tunnet_{nname}", interfaces=[fnet_iface], type="IPv4")
        n.add_route(subnet=fablib.FABNETV4_SUBNET, next_hop=fnet.get_gateway())
        return n, fnet_iface

    tun_node = {}          # 'out'/'in' -> node (same node for both if not TUNNEL_SPLIT)
    tun_fabnet_iface = {}  # 'out'/'in' -> that node's FABNet interface
    if TUNNEL_SPLIT:
        if out_links:
            tun_node["out"], tun_fabnet_iface["out"] = make_tunnel_node(tunnel_node_name(letter, "out"))
        if in_links:
            tun_node["in"], tun_fabnet_iface["in"] = make_tunnel_node(tunnel_node_name(letter, "in"))
    else:
        if out_links or in_links:
            node, fnet_iface = make_tunnel_node(tunnel_node_name(letter, "out"))
            tun_node["out"] = tun_node["in"] = node
            tun_fabnet_iface["out"] = tun_fabnet_iface["in"] = fnet_iface

    # Source side: switch.out_cross -> tunnel data NIC x{vni} (role="src")
    for l in out_links:
        vni = l["vni"]
        rank, idx = l["src_switch"]
        tun = tun_node["out"]
        data_if = tun.add_component(model="NIC_Basic", name=f"x{vni}").get_interfaces()[0]
        data_if.set_mode("manual")
        sw_if = switches[(rank, idx)].get_port_interface("out_cross")
        sw_if.set_mode("manual")
        s.add_monitored_l3network(
            name=f"lx{l['i']}", interfaces=[sw_if, data_if], sinks=[data_if],
            fabnet_iface=tun_fabnet_iface["out"], vni=vni, role="src", site=site,
        )

    # Destination side: tunnel data NIC x{vni} -> switch.in_cross (role="dst")
    for l in in_links:
        vni = l["vni"]
        rank, idx = l["dst_switch"]
        tun = tun_node["in"]
        data_if = tun.add_component(model="NIC_Basic", name=f"x{vni}").get_interfaces()[0]
        data_if.set_mode("manual")
        sw_if = switches[(rank, idx)].get_port_interface("in_cross")
        sw_if.set_mode("manual")
        s.add_monitored_l3network(
            name=f"lx{l['i']}_d", interfaces=[data_if, sw_if], sinks=[sw_if],
            fabnet_iface=tun_fabnet_iface["in"], vni=vni, role="dst", site=site,
        )

    print(f"Built slice {slice_name} ({site}): {len(s.get_nodes())} experiment nodes + "
          f"monitors/analyzer ({len(out_links)} outgoing / {len(in_links)} incoming "
          f"cross-site links).")
    return {
        "slice": s, "site": site, "switches": switches, "senders": senders,
        "receivers": receivers, "ctl": ctl, "ctrl_net": ctrl_net, "tun_node": tun_node,
    }

slice_handles = {"A": build_slice("A"), "B": build_slice("B")}
print("Both slices built. Ready to submit.")

## 4. Submit both slices

Crinkle's `submit()` places monitors on different workers from the nodes they watch, then runs post-boot config (this is the slow step). Submitting two half-size slices (instead of one full-size slice) is what keeps each request under FABRIC's orchestrator payload limit. The two submits run as parallel threads; the cell waits for both and reports every exception (progress output from the two slices will interleave).

In [ ]:
# The orchestrator client's default HTTP timeout (30 s) is shorter than the
# time a ~60-node create takes to return, and urllib3 silently retries the
# timed-out POST -- the retry then fails against the slice the first request
# actually created. Raise the timeout and disable HTTP retries.
from urllib3.util.retry import Retry

oc = fablib.get_manager().orch
oc.timeout = 300
sess = getattr(oc, "session", None) or getattr(oc, "_session")
for prefix in ("https://", "http://"):
    sess.adapters[prefix].max_retries = Retry(
        total=0, connect=0, read=0, status=0, redirect=0)
print(f"orchestrator timeout={oc.timeout}s, HTTP retries disabled")

In [ ]:
import traceback

def submit_slice(letter):
    s = slice_handles[letter]["slice"]
    s.submit()
    print(f"Slice {s.get_name()} active.")

submit_errors = {}
with futures.ThreadPoolExecutor(max_workers=2) as pool:
    jobs = {pool.submit(submit_slice, letter): letter for letter in slice_handles}
    for job in futures.as_completed(jobs):
        letter = jobs[job]
        exc = job.exception()
        if exc is not None:
            submit_errors[letter] = exc

for letter, exc in submit_errors.items():
    print(f"\n--- Slice {letter} submit FAILED ---")
    traceback.print_exception(exc)

if submit_errors:
    raise RuntimeError(
        "slice submit failed for: "
        + ", ".join(f"{letter} ({type(exc).__name__})"
                    for letter, exc in submit_errors.items())
    )
print("Both slices active.")

Once both slices are active, refresh every cached node handle from its
live slice via `slice.get_node()`. After `submit()` the slice inventory is
authoritative (post-boot management IPs, device names, etc.), so the build-time
node objects stored in `slice_handles` are replaced with freshly-fetched ones.
(BMv2 switches are refreshed separately with `get_attestable_switch()` in the
discover cell, since `get_node()` would drop the switch-specific methods.)

In [ ]:
for letter, h in slice_handles.items():
    s = h["slice"]
    for sid in h["senders"]:
        h["senders"][sid] = s.get_node(name=snd_name(sid))
    for r in h["receivers"]:
        h["receivers"][r] = s.get_node(name=rcv_name(r))
    h["ctl"] = s.get_node(name=h["ctl"].get_name())
    # tun_node's 'out'/'in' may point at the same physical node (no TUNNEL_SPLIT);
    # fetch each distinct name once so shared entries keep a shared handle.
    refreshed = {}
    for direction, node in h["tun_node"].items():
        nm = node.get_name()
        h["tun_node"][direction] = refreshed.setdefault(nm, s.get_node(name=nm))
    print(f"slice {letter}: refreshed {len(h['senders'])} senders, "
          f"{len(h['receivers'])} receivers, 1 controller, "
          f"{len(refreshed)} tunnel node(s)")
print("all stored node handles refreshed from the live slices")

## 5. Bring up the cross-site VXLAN tunnels

Both slices are now live in the kernel, so every tunnel node's FABNet IP is
known. For each of the `WIDTH` directed cross-site links: create a VXLAN
netdev on the source tunnel node addressed to the destination tunnel node's
FABNet IP (and the matching VXLAN netdev on the destination side addressed
back), then bridge each tunnel node's pre-encap/post-decap data NIC
(`x{vni}`, the local half of the `add_monitored_l3network` link) to that
VXLAN netdev. This is the only place raw sockets/VXLAN details appear; no
DiDAQt or BMv2 code is touched.

In [ ]:
def tunnel_node_handle(letter, direction):
    hs = slice_handles[letter]
    name = tunnel_node_name(letter, direction)
    return hs["slice"].get_node(name=name)

def bring_up_vxlan(node, vni, fabnet_dev, local_ip, remote_ip, data_dev):
    # The FABNet dev must carry inner frame (up to 1500 with the monitor
    # UID trailer) + ~70 bytes of IPv6 VXLAN overhead; at the default 1500
    # the kernel derives vxlan MTU 1450 and the bridge drops every data
    # frame. FABRIC dataplane supports 9000.
    cmd = (
        f"sudo ip link set {fabnet_dev} mtu 9000; "
        f"sudo ip link add vxlan{vni} type vxlan id {vni} "
        f"remote {remote_ip} local {local_ip} dstport 4789 dev {fabnet_dev}; "
        f"sudo ip link set vxlan{vni} mtu 1600; "
        f"sudo ip link set vxlan{vni} up; "
        f"sudo ip link add br{vni} type bridge; "
        f"sudo ip link set {data_dev} master br{vni}; "
        f"sudo ip link set vxlan{vni} master br{vni}; "
        f"sudo ip link set br{vni} up; "
        f"sudo ip link set {data_dev} up"
    )
    return node.execute_thread(cmd)

vxlan_jobs = []
for l in CROSS_LINKS:
    vni = l["vni"]
    src_tun = tunnel_node_handle(l["src_letter"], "out")
    dst_tun = tunnel_node_handle(l["dst_letter"], "in")

    src_fnet_if = src_tun.get_interface(network_name=f"tunnet_{src_tun.get_name()}")
    dst_fnet_if = dst_tun.get_interface(network_name=f"tunnet_{dst_tun.get_name()}")
    src_ip, dst_ip = src_fnet_if.get_ip_addr(), dst_fnet_if.get_ip_addr()
    src_fdev, dst_fdev = src_fnet_if.get_device_name(), dst_fnet_if.get_device_name()

    src_ddev = src_tun.get_component(name=f"x{vni}").get_interfaces()[0].get_device_name()
    dst_ddev = dst_tun.get_component(name=f"x{vni}").get_interfaces()[0].get_device_name()

    vxlan_jobs.append(bring_up_vxlan(src_tun, vni, src_fdev, src_ip, dst_ip, src_ddev))
    vxlan_jobs.append(bring_up_vxlan(dst_tun, vni, dst_fdev, dst_ip, src_ip, dst_ddev))

for j in futures.as_completed(vxlan_jobs):
    j.result()
print(f"VXLAN tunnels up for {len(CROSS_LINKS)} cross-site links "
      f"({len(vxlan_jobs)} tunnel endpoints configured).")

# Sanity check: confirm the fork's L3-monitoring metadata round-trips through
# the live slices (exercises CrinkleSlice.get_monitored_l3networks()).
l3_a = slice_handles["A"]["slice"].get_monitored_l3networks()
l3_b = slice_handles["B"]["slice"].get_monitored_l3networks()
print(f"Crinkle L3-monitored links found: {len(l3_a)} in {SLICE_A}, {len(l3_b)} in {SLICE_B} "
      f"(expect {len(CROSS_LINKS)} 'src' + {len(CROSS_LINKS)} 'dst' halves total)")

## 6. Discover MACs / devices and compute forwarding rules

After boot we read each receiver's **data NIC MAC** (the routing key every
sender carries in its frames' *source* field -- destinations are broadcast,
see below) and each switch port's OS device name, then compute the
per-switch `l2_forward` table entries from the butterfly self-routing rule.
Handles from both slices are merged into flat dicts keyed by the global
`(rank, idx)` / sender id / receiver index — every switch already gets rules
for **every** receiver (not just its initial target), which is what lets the
cross-site probe (section 11) work without any BMv2 rule changes.

Frames are **broadcast-addressed**: FABRIC SR-IOV NICs silently drop
unicast frames whose destination MAC they do not own, so a receiver MAC in
the destination field dies at the first hop whose NIC has not been assigned
that MAC. The sender's `-b` flag therefore sends to `ff:ff:ff:ff:ff:ff` and
puts the routing MAC in the *source* field; `l2_forward_bmv2.p4` matches on
`src_addr`. Rule contents and failover semantics are unchanged -- only the
matched header field moves.


In [ ]:
# Refresh handles from the live slices (two independent FABRIC slice objects).
slices = {"A": slice_handles["A"]["slice"], "B": slice_handles["B"]["slice"]}

switches = {(rank, idx): slices[slice_letter_of(idx)].get_attestable_switch(name=sw_name(rank, idx))
            for rank in range(SWITCH_RANKS) for idx in range(WIDTH)}
receivers = {r: slices[slice_letter_of(r)].get_node(name=rcv_name(r)) for r in range(NUM_RECEIVERS)}
senders   = {sid: slices[slice_letter_of(pair_of_sender(sid))].get_node(name=snd_name(sid))
             for sid in range(1, NUM_SENDERS + 1)}
ctl_main   = slices["A"].get_node(name="ctlmain")
ctl_follow = slices["B"].get_node(name="ctlfollow")

# Receiver data-plane MACs + devices.
rx_mac = {}      # r -> data NIC MAC (the frames' source-field routing key)
rx_dev = {}      # r -> data NIC OS device
for r in range(NUM_RECEIVERS):
    dif = receivers[r].get_component(name="d").get_interfaces()[0]
    rx_mac[r] = dif.get_mac()
    rx_dev[r] = dif.get_device_name()

# Sender data-plane devices + the receiver each sender targets.
snd_dev = {}     # sid -> data NIC OS device
snd_target = {}  # sid -> receiver index (== its pair index)
for sid in range(1, NUM_SENDERS + 1):
    snd_dev[sid] = senders[sid].get_component(name="d").get_interfaces()[0].get_device_name()
    snd_target[sid] = pair_of_sender(sid)

# Control-plane IPs (each controller lives on its own slice's ctlnet_X).
ctl_main_ip   = ctl_main.get_interface(network_name="ctlnet_A").get_ip_addr()
ctl_follow_ip = ctl_follow.get_interface(network_name="ctlnet_B").get_ip_addr()

# Per-switch forwarding rules: list of {"route_mac", "port"} keyed by (rank, idx).
# NOTE: rules cover every receiver, not just the initial mapping -- this is
# what the cross-site probe relies on (no BMv2 rule changes needed there).
sw_rules = {}
for rank in range(SWITCH_RANKS):
    for idx in range(WIDTH):
        ports = switch_ports(rank)
        rules = []
        for r in range(NUM_RECEIVERS):
            eidx = egress_index(rank, idx, r, ports)
            if eidx is not None:
                rules.append({"route_mac": rx_mac[r], "port": eidx})
        sw_rules[(rank, idx)] = rules

print("Receiver data MACs:")
for r in range(NUM_RECEIVERS):
    print(f"  rcv{r} ({site_of(r)}): {rx_mac[r]}  dev={rx_dev[r]}")
print(f"main controller ip (site A):     {ctl_main_ip}")
print(f"follower controller ip (site B): {ctl_follow_ip}")
print(f"example rules for sw0n0: {sw_rules[(0,0)]}")

## 7. Program the BMv2 switches

For each switch: start `simple_switch`, hot-load the compiled `l2_forward_bmv2.p4`,
then install its forwarding entries via `simple_switch_CLI` (`run_command`). All
switches from **both** slices are configured in parallel (they're just fablib
node handles from `switches`, merged above — which slice each one lives in
doesn't matter here).

In [ ]:
def mac_to_hex(mac):
    return "0x" + mac.replace(":", "").lower()

def program_switch(key):
    rank, idx = key
    sw = switches[key]
    name = sw_name(rank, idx)
    # 1) launch the (empty) simple_switch bound to this switch's ports
    sw.start_switch()
    # 2) compile + hot-swap our L2 program
    if not sw.load_program(P4_LOCAL):
        return name, False, "load_program failed"
    # 3) install forwarding entries
    installed = 0
    for rule in sw_rules[key]:
        cmd = f"table_add {P4_TABLE} forward {mac_to_hex(rule['route_mac'])} => {rule['port']}"
        if sw.run_command(cmd):
            installed += 1
    return name, True, f"{installed}/{len(sw_rules[key])} rules"

results = {}
with futures.ThreadPoolExecutor(max_workers=16) as pool:
    jobs = {pool.submit(program_switch, key): key for key in switches}
    for job in futures.as_completed(jobs):
        name, ok, msg = job.result()
        results[name] = (ok, msg)
        print(f"  {name}: {'OK' if ok else 'FAIL'} — {msg}")

# Sanity: dump one switch's tables so you can confirm the table name / entries.
print("\n--- sw0n0 tables ---")
switches[(0, 0)].run_command("show_tables")
switches[(0, 0)].run_command(f"table_dump {P4_TABLE}")

## 8. Build DiDAQt on the end hosts and generate the topology YAML

Upload the repo to the senders, receivers and controllers (across **both**
slices) and `make examples`. The controller runs in **log-only** mode for this
milestone (no switch control channel yet).

**One shared `topology.yaml`**, describing the whole logical butterfly graph
(all switches/senders/receivers, regardless of which physical site hosts
them), is uploaded to **both** controllers — exactly as in the single-slice
version. The two-slice split is a FABRIC deployment detail; DiDAQt's HA
controller model still expects one full topology so either the main or the
follower can run path-finding/failover over the complete graph. A switch's
`out_cross`/`in_cross` port at the site boundary still just names its peer
switch in `topology.yaml` — the fact that a VXLAN tunnel physically realizes
that link is invisible at this level.

In [ ]:
end_hosts = ([senders[s] for s in senders] +
             [receivers[r] for r in receivers] +
             [ctl_main, ctl_follow])

# --- deps ---
install_cmd = ("sudo apt-get update -qq && "
               "sudo apt-get install -y -qq build-essential ethtool libyaml-dev")
jobs = [n.execute_thread(install_cmd) for n in end_hosts]
for j in futures.as_completed(jobs):
    j.result()
print("deps installed")

# --- upload repo + build ---
# upload_directory() extracts into <dst>/<basename(src)>, so upload to the
# parent of REMOTE_DIR (their basenames must match for the repo to land there).
assert os.path.basename(REPO) == os.path.basename(REMOTE_DIR)
jobs = [n.upload_directory_thread(REPO, os.path.dirname(REMOTE_DIR)) for n in end_hosts]
for j in futures.as_completed(jobs):
    j.result()
# node.execute() never raises on a failed command, so verify the binaries exist.
jobs = {n.execute_thread(f"cd {REMOTE_DIR} && make examples >/dev/null && ls build/sender"): n
        for n in end_hosts}
failed = [jobs[j].get_name() for j in futures.as_completed(jobs)
          if "sender" not in j.result()[0]]
if failed:
    raise RuntimeError(f"didaqt build failed on: {', '.join(sorted(failed))}")
print("didaqt built on end hosts")

In [ ]:
# Generate a butterfly topology.yaml for the DiDAQt controller.
# Schema follows examples/topology.yaml: a sequence of node maps with
# connections (keyed by local port) and initial_connections ({sender,receiver}).
import io

def yaml_conn(local_port, other_node, other_port, flows=None, bw="100G"):
    s = f"    {local_port}:\n"
    s += f"      other_node: {other_node}\n"
    s += f"      other_port: {other_port}\n"
    s += f"      max_bandwidth: {bw}\n"
    if flows:
        s += "      initial_connections:\n"
        for i, (snd, rcv) in enumerate(flows, 1):
            s += f"        {i}: {{ sender: {snd}, receiver: {rcv} }}\n"
    return s

# Precompute, for every switch, the straight/cross next hop and the set of
# initial flows crossing each port (initial routing = each sender -> its receiver
# along the self-routed straight/cross path).
def route_path(recv_idx):
    """Return the ordered list of (rank, idx) switches a frame to recv_idx takes,
    entering at rank-0 switch = recv_idx's pair (its own initial rank-0 node)."""
    # A sender in pair p targets receiver p, entering rank-0 node p.
    node = recv_idx  # initial rank-0 node index for this receiver's senders
    path = [(0, node)]
    for s in range(K):
        b = s
        if bit(node, b) != bit(recv_idx, b):
            node ^= (1 << b)
        path.append((s + 1, node))
    return path

# flows_on[(rank,idx)][port_name] = list of (sender_name, receiver_name)
from collections import defaultdict
flows_on = defaultdict(lambda: defaultdict(list))
for sid in range(1, NUM_SENDERS + 1):
    r = snd_target[sid]
    path = route_path(r)
    for hop in range(len(path) - 1):
        rank, idx = path[hop]
        nxt_rank, nxt_idx = path[hop + 1]
        port = "out_str" if nxt_idx == idx else "out_cross"
        flows_on[(rank, idx)][port].append((snd_name(sid), rcv_name(r)))
    # last hop into the receiver
    lr, li = path[-1]
    flows_on[(lr, li)]["out_rx"].append((snd_name(sid), rcv_name(r)))

buf = io.StringIO()
# senders
for sid in range(1, NUM_SENDERS + 1):
    p = pair_of_sender(sid)
    ports = switch_ports(0)
    in_port = 1 + (sid - 1) % SENDERS_PER_SW      # switch-side port number (1-based)
    buf.write(f"- name: {snd_name(sid)}\n")
    buf.write("  type: sender\n")
    buf.write(f"  sender_id: {sid}\n")
    buf.write("  sender_id_bytes: 26\n")
    buf.write("  max_bandwidth: 10G\n")
    buf.write(f"  initial_receiver: {rcv_name(snd_target[sid])}\n")
    buf.write(f"  group_id: {p + 1}\n")
    buf.write("  connections:\n")
    buf.write(yaml_conn(1, sw_name(0, p), in_port,
                        flows=[(snd_name(sid), rcv_name(snd_target[sid]))]))
# switches
for rank in range(SWITCH_RANKS):
    for idx in range(WIDTH):
        ports = switch_ports(rank)
        buf.write(f"- name: {sw_name(rank, idx)}\n")
        buf.write("  type: switch\n")
        buf.write("  switch_type_group: bmv2\n")
        buf.write("  connections:\n")
        for pnum, pname in enumerate(ports, 1):
            # resolve peer for this port
            if pname.startswith("in_s") and pname[4:].isdigit():
                j = int(pname[4:]); sid = idx * SENDERS_PER_SW + j + 1
                other, oport = snd_name(sid), 1
            elif pname == "in_str":
                other, oport = sw_name(rank - 1, idx), 1 + switch_ports(rank - 1).index("out_str")
            elif pname == "in_cross":
                src = idx ^ cross_bit(rank - 1)
                other, oport = sw_name(rank - 1, src), 1 + switch_ports(rank - 1).index("out_cross")
            elif pname == "out_str":
                other, oport = sw_name(rank + 1, idx), 1 + switch_ports(rank + 1).index("in_str")
            elif pname == "out_cross":
                dst = idx ^ cross_bit(rank)
                other, oport = sw_name(rank + 1, dst), 1 + switch_ports(rank + 1).index("in_cross")
            elif pname == "out_rx":
                other, oport = rcv_name(idx), 1
            buf.write(yaml_conn(pnum, other, oport, flows=flows_on[(rank, idx)].get(pname)))
# receivers
for r in range(NUM_RECEIVERS):
    buf.write(f"- name: {rcv_name(r)}\n")
    buf.write("  type: receiver\n")
    buf.write(f"  receiver_id: {r}\n")
    buf.write("  connections:\n")
    inflows = flows_on[(K, r)].get("out_rx")
    buf.write(yaml_conn(1, sw_name(K, r), 1 + switch_ports(K).index("out_rx"), flows=inflows))

topo_yaml = buf.getvalue()
local_topo = os.path.join(RUN_DIR.replace("/tmp", "."), "topology.yaml")
os.makedirs(os.path.dirname(local_topo), exist_ok=True)
with open(local_topo, "w") as f:
    f.write(topo_yaml)
print(topo_yaml[:1200])
print("... (truncated)")

# upload the SAME topology.yaml to both controllers (main in site A, follower in site B)
for c in (ctl_main, ctl_follow):
    c.execute(f"mkdir -p {REMOTE_DIR}", quiet=True)
    c.upload_file(local_topo, f"{REMOTE_DIR}/topology.yaml")
print("topology.yaml uploaded to both controllers")

## 9. Bring up data-plane interfaces & start the Crinkle monitors

The sender/receiver data NICs are raw L2; bring them up (+promisc, VLAN offload off). Then start the inline DPDK monitors on **both** slices: every butterfly link (intra-site L2 and the local half of each cross-site L3 link) runs *through* a Crinkle monitor, which is a bump-in-the-wire forwarder — until its DPDK app is running the link passes no traffic, so this must happen before any workload.

In [ ]:
def bringup(node, comp="d"):
    dev = node.get_component(name=comp).get_interfaces()[0].get_device_name()
    cmd = (f"sudo ip link set {dev} up; "
           f"sudo ip link set {dev} promisc on; "
           f"sudo ethtool -K {dev} txvlan off rxvlan off 2>/dev/null; true")
    return node.execute_thread(cmd)

jobs = [bringup(senders[s]) for s in senders] + [bringup(receivers[r]) for r in receivers]
for j in futures.as_completed(jobs):
    j.result()
print("data-plane interfaces up")

# Start the inline Crinkle DPDK monitors on BOTH slices. Each monitor forwards
# vport1<->vport2 only while its DPDK app runs; without this, every monitored
# link (and thus the whole data plane) is dead. Covers intra-site L2 monitors
# and the local-half monitors of the cross-site L3 links (both are in
# slice.monitors).
for letter in ("A", "B"):
    slice_handles[letter]["slice"].start_all_monitors(wait=True)
print("Crinkle monitors started on both slices")

## 10. Run the reachability test + cross-site probe

Launch, in order, the **controllers** (main in site A, follower in site B —
HA pair, coordinating over FABNet), the **receivers** (both sites, all
reporting heartbeats to the main controller), and the **senders** (both
sites, each emitting the rate-limited workload routed to its receiver via `-b` --
broadcast destination, receiver MAC as the source-field routing key -- for
`TEST_SECS`). Then run a short **cross-site probe** burst: two extra sender
processes point their frames at a receiver in the *other* site
(`PROBE_AB_SENDER` -> `PROBE_AB_RECEIVER`, `PROBE_BA_SENDER` ->
`PROBE_BA_RECEIVER`). No BMv2 rule changes are needed for this — every switch
was already programmed with forwarding rules for every receiver (see the
discover cell) — so simply setting a sender's routing MAC to a different
receiver routes its frames along that receiver's normal self-routed path, which
in this case crosses the site boundary through the VXLAN tunnel. Output is
redirected to per-node log files under `RUN_DIR`.

In [ ]:
def sh(node, cmd):
    return node.execute_thread(f"mkdir -p {RUN_DIR}; {cmd}")

# 1) controllers (log-only DiDAQt controller wrapped by the HA script)
ctl_cmd = f"sudo ./build/controller topology.yaml {HB_PORT}"
sh(ctl_main,
   f"cd {REMOTE_DIR} && sudo nohup python3 ./artifact/controller_ha.py --role main "
   f"--peer {ctl_follow_ip} --hb-port {HA_PORT} -- {ctl_cmd} "
   f"> {RUN_DIR}/ctl_main.log 2>&1 &")
sh(ctl_follow,
   f"cd {REMOTE_DIR} && sudo nohup python3 ./artifact/controller_ha.py --role follower "
   f"--peer {ctl_main_ip} --hb-port {HA_PORT} -- {ctl_cmd} "
   f"> {RUN_DIR}/ctl_follow.log 2>&1 &")
import time
time.sleep(3)

# 2) receivers (both sites)
for r in range(NUM_RECEIVERS):
    sh(receivers[r],
       f"cd {REMOTE_DIR} && sudo nohup ./build/receiver {rx_dev[r]} {r} "
       f"{ctl_main_ip} {HB_PORT} > {RUN_DIR}/rcv{r}.log 2>&1 &")
time.sleep(2)

# 3) senders (both sites, bounded initial-mapping workload)
for sid in range(1, NUM_SENDERS + 1):
    r = snd_target[sid]
    sh(senders[sid],
       f"cd {REMOTE_DIR} && sudo nohup timeout {TEST_SECS} ./build/sender "
       f"-b -r {SEND_RATE} -l {SEND_LEN} {snd_dev[sid]} {rx_mac[r]} {sid} "
       f"> {RUN_DIR}/snd{sid}.log 2>&1 &")
print(f"workload running for ~{TEST_SECS}s ...")
time.sleep(TEST_SECS + 5)

# 4) cross-site probe burst: same senders, briefly re-pointed at a receiver in
# the OTHER site. This is the only traffic in this notebook that actually
# crosses a VXLAN tunnel -- the initial mapping above never does (sender pair
# p -> receiver p share the same index, hence the same half, by construction).
for sid, r in ((PROBE_AB_SENDER, PROBE_AB_RECEIVER), (PROBE_BA_SENDER, PROBE_BA_RECEIVER)):
    sh(senders[sid],
       f"cd {REMOTE_DIR} && sudo nohup timeout {PROBE_SECS} ./build/sender "
       f"-b -r {SEND_RATE} -l {SEND_LEN} {snd_dev[sid]} {rx_mac[r]} {sid} "
       f"> {RUN_DIR}/probe_snd{sid}.log 2>&1 &")
print(f"cross-site probe running for ~{PROBE_SECS}s "
      f"(snd{PROBE_AB_SENDER} -> rcv{PROBE_AB_RECEIVER}, snd{PROBE_BA_SENDER} -> rcv{PROBE_BA_RECEIVER}) ...")
time.sleep(PROBE_SECS + 5)

# stop everything (leave BMv2 switches + VXLAN tunnels running)
for n in end_hosts:
    n.execute("sudo killall -q sender receiver controller heartbeat_monitor; "
              "sudo pkill -f controller_ha; true", quiet=True)
print("workload + probe stopped")

## 11. Verify reachability + cross-site probe

Two checks, printed separately:

**(a) Per-slice initial-mapping reachability** — pull each receiver's log,
strip the ANSI TUI, and confirm each receiver saw **valid** frames from
**both** of its assigned senders (as in the single-slice notebook). This
check alone is intra-site by construction and does **not** exercise the new
FABNet/VXLAN cross-site links.

**(b) Cross-site probe** — confirm the far-site receiver actually saw valid
frames from the probe sender (`PROBE_AB_SENDER` at `PROBE_AB_RECEIVER` and
`PROBE_BA_SENDER` at `PROBE_BA_RECEIVER`). This is the check that proves a
VXLAN tunnel carried real butterfly frames end to end.

In [ ]:
import re
ANSI = re.compile(r"\x1b\[[0-9;]*[A-Za-z]")

def strip_ansi(s):
    return ANSI.sub("", s)

def expected_senders(r):
    return [r * SENDERS_PER_SW + j + 1 for j in range(SENDERS_PER_SW)]

def read_valid_count(log_text, sender_id):
    """Find the row for sender_id in a receiver's (ANSI-stripped) TUI log and
    return its 'valid' count (0 if never seen)."""
    valid = 0
    for line in log_text.splitlines():
        toks = line.split()
        if toks and toks[0] == str(sender_id):
            nums = [t for t in toks[1:] if t.isdigit()]
            if nums:
                valid = int(nums[0])
    return valid

# --- (a) per-slice initial-mapping reachability ---
overall_ok = True
print("(a) initial-mapping reachability (intra-site by construction)")
print(f"{'receiver':10} {'sender':8} {'valid':>10} {'status'}")
rcv_logs = {}
for r in range(NUM_RECEIVERS):
    out, _ = receivers[r].execute(f"cat {RUN_DIR}/rcv{r}.log", quiet=True)
    rcv_logs[r] = strip_ansi(out)
    for sid in expected_senders(r):
        valid = read_valid_count(rcv_logs[r], sid)
        ok = valid > 0
        overall_ok &= ok
        flag = "" if ok else "   <-- NO FRAMES"
        print(f"rcv{r:<7} snd{sid:<5} {valid:>10} {'OK' if ok else 'MISSING'}{flag}")
print("\n=== (a) REACHABILITY: " + ("PASS ===" if overall_ok else "FAIL ==="))

# --- (b) cross-site probe ---
print("\n(b) cross-site probe (proves the VXLAN tunnel carries butterfly frames)")
def check_probe(sender_sid, receiver_idx, label):
    out, _ = receivers[receiver_idx].execute(f"cat {RUN_DIR}/rcv{receiver_idx}.log", quiet=True)
    valid = read_valid_count(strip_ansi(out), sender_sid)
    ok = valid > 0
    print(f"  {label}: snd{sender_sid} ({site_of(pair_of_sender(sender_sid))}) -> "
          f"rcv{receiver_idx} ({site_of(receiver_idx)})  valid={valid}  {'PASS' if ok else 'FAIL'}")
    return ok

probe_ab_ok = check_probe(PROBE_AB_SENDER, PROBE_AB_RECEIVER, "A->B")
probe_ba_ok = check_probe(PROBE_BA_SENDER, PROBE_BA_RECEIVER, "B->A")
probe_ok = probe_ab_ok and probe_ba_ok
print("\n=== (b) CROSS-SITE PROBE: " + ("PASS ===" if probe_ok else "FAIL ==="))

print("\n=== OVERALL: " + ("PASS ===" if (overall_ok and probe_ok) else "FAIL ==="))
print("\n--- main controller (log-only) tail ---")
out, _ = ctl_main.execute(f"cat {RUN_DIR}/ctl_main.log", quiet=True)
print(strip_ansi(out)[-1500:])

## 12. Failover experiment -- setup

DiDAQt failover, timed end to end.

- **Decision plane**: the DiDAQt controller (HA pair under `controller_ha.py`)
  processes receiver heartbeats and runs the failover state machine.
- **Actuation, out of band**: the controller commands switches over the FABRIC
  **management plane** (ICMPv6 echo, id `0xDDAA`) -- TCP/UDP between nodes is
  blocked there but ICMP passes. Every BMv2 switch node runs
  `artifact/bmv2_switch_agent.py` (persistent `simple_switch_CLI` backend, so updates are
  ms-scale); the controller is started with `-s switches.map` (switch name ->
  management IPv6) so each `UPDATE` reaches the right agent. Agents ACK
  best-effort: a NACK makes didaqt_ctrl roll the failover back and retry forever
  when the dead switch is on the failed path itself -- dead switches are instead
  detected by the data-plane heartbeats, which is DiDAQt's model.
- **Controller HA**: receivers heartbeat to a small UDP relay on the main
  controller's *node* (port 9001) which duplicates each datagram to the local
  controller and the follower. Killing the main controller *process* leaves the
  relay up, so the follower's controller (launched at takeover) is fed
  immediately. A fresh follower re-derives sender placement from heartbeats:
  expect one convergence failover cycle per previously-moved sender before the
  next fault is injected.
- **Failover target order is structural**: path preference is contention (tied in
  a full butterfly), then hop count (tied at 4), then DFS order -- and DFS varies
  the deepest stage first, which is the site-crossing stage. The FIRST alternative
  for every sender is therefore the other site's sibling receiver (`rcv p^4`).
  A cross-site failover needs only a single path fault (section 14); an intra-site
  failover requires the sibling path to be dead too, which killing the sender's
  rank-2 switch achieves (section 13: failover #1 to the sibling stays dark, grace
  expires, failover #2 lands intra-site).
- **Timing**: `t_fail` is stamped on the fault node in the same SSH exec as the
  fault; `t_appear` is the first frame at the new receiver (`tcpdump -tt -c 1`
  matching the sender's routing MAC in the frame *source* field). Recovery is
  their difference across NTP-synced clocks (the setup cell prints measured
  offsets). Controller internals (`t_decision`, per-switch `rtt`, `t_confirm`)
  come from its event log.
- **Provenance**: SPADE stores flow fields, not MACs, so the sender's flow is
  filtered by `ip.src == 10.0.<sid>.1` -- the working equivalent of its constant
  source/routing MAC. Both slices' analyzers are queried and merged by packet UID.

Requires the didaqt repo on this JupyterHub to include the agent-map controller
changes (`controller.c -s`, handler switch names) and `artifact/bmv2_switch_agent.py`.

In [ ]:
import ipaddress, re, time

# ---- Experiment parameters ----
EXP_RATE      = 100     # frames/sec/sender: low, minimal BMv2 droppage
EXP_HB_MS     = 250     # receiver heartbeat interval (25 frames/interval)
EXP_MISS      = 3       # missed heartbeats before failover (~0.75-1s detection)
EXP_GRACE_MS  = 1500    # post-failover grace (actuation is ms-scale ICMPv6)
EXP_SEND_SECS = 1800    # sender lifetime per workload start
RELAY_PORT    = 9001    # receivers -> relay on ctl_main node -> both controllers
AGENT_CONF    = "/tmp/bmv2_switch_agent.conf"
AGENT_LOG     = "/tmp/bmv2_switch_agent.log"

EXP_ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
def sansi(s):
    return EXP_ANSI.sub("", s or "")

# Routing MAC per sender: the constant table key (initial receiver's data MAC).
routing_mac = {sid: rx_mac[pair_of_sender(sid)] for sid in senders}

# The hub-local repo must carry the agent-map support before syncing it out.
assert "agent_map" in open(os.path.join(REPO, "examples/controller.c")).read(), \
    "sync the didaqt repo first: controller.c lacks -s agent_map support"
assert os.path.exists(os.path.join(REPO, "artifact/bmv2_switch_agent.py")), \
    "sync the didaqt repo first: artifact/bmv2_switch_agent.py missing"

# ---- 0) ctl_follow ctlnet IP (fe80 = never configured; HA + relay need it) ----
if ctl_follow_ip.lower().startswith("fe80"):
    netB = slice_handles["B"]["slice"].get_network(name="ctlnet_B")
    iface = ctl_follow.get_interface(network_name="ctlnet_B")
    addr = netB.get_available_ips()[0]
    iface.ip_addr_add(addr=addr, subnet=netB.get_subnet())
    ctl_follow.ip_route_add(subnet=ipaddress.ip_network("10.128.0.0/10"),
                            gateway=netB.get_gateway())
    ctl_follow_ip = str(addr)
print(f"controller ctlnet IPs (receiver heartbeats + HA over FABNet IPv4): "
      f"main={ctl_main_ip} follower={ctl_follow_ip}")

# ---- 1) rebuild controllers with the agent-map support ----
for f in ("include/didaqt.h", "src/didaqt_ctrl.c", "examples/controller.c"):
    for n in (ctl_main, ctl_follow):
        n.upload_file(os.path.join(REPO, f), f"{REMOTE_DIR}/{f}")
jobs = {n.execute_thread(f"cd {REMOTE_DIR} && make -B examples >/dev/null && "
                         f"./build/controller 2>&1 | head -1"): n.get_name()
        for n in (ctl_main, ctl_follow)}
for j in futures.as_completed(jobs):
    out = j.result()[0]
    assert "-s agent_map" in out, f"{jobs[j]}: rebuild lacks -s ({out!r})"
print("controllers rebuilt with agent-map support")

# ---- 2) switch name -> management IPv6 map, uploaded to both controllers ----
switch_mgmt = {sw_name(*k): str(switches[k].get_management_ip()) for k in switches}
assert all(":" in ip for ip in switch_mgmt.values()), \
    f"expected IPv6 management addresses: {switch_mgmt}"
# switch commands travel the management plane as ICMPv6, so the
# controllers need IPv6 management addresses to source them
for tag, n in (("ctl_main", ctl_main), ("ctl_follow", ctl_follow)):
    mip = str(n.get_management_ip())
    assert ":" in mip, f"{tag} management IP is not IPv6: {mip}"
local_map = os.path.join(RUN_DIR.replace("/tmp", "."), "switches.map")
with open(local_map, "w") as f:
    f.write("".join(f"{n} {ip}\n" for n, ip in sorted(switch_mgmt.items())))
for n in (ctl_main, ctl_follow):
    n.upload_file(local_map, f"{REMOTE_DIR}/switches.map")
print(f"switches.map uploaded ({len(switch_mgmt)} agents)")

# ---- 3) deploy + start a bmv2_switch_agent on every switch node ----
agent_local = os.path.join(REPO, "artifact/bmv2_switch_agent.py")
conf_common = "".join(f"sender {sid} {routing_mac[sid]}\n" for sid in sorted(senders))

def deploy_agent(key):
    sw = switches[key]
    sw.upload_file(agent_local, "/tmp/bmv2_switch_agent.py")
    conf = f"name {sw_name(*key)}\nthrift_port 9090\ntable {P4_TABLE}\n" + conf_common
    # pkill gets its own exec: the deploy command below contains the agent
    # path literally, so a combined command would pkill its own shell.
    sw.execute("sudo pkill -f '[b]mv2_switch_agent.py' 2>/dev/null; true", quiet=True)
    out, _ = sw.execute(
        f"cat > {AGENT_CONF} << 'EOF'\n{conf}EOF\n"
        f"rm -f /tmp/bmv2_switch_agent_stop; "
        f"sudo nohup python3 -u /tmp/bmv2_switch_agent.py {AGENT_CONF} > {AGENT_LOG} 2>&1 & "
        f"sleep 3; grep -c 'listening for ICMPv6' {AGENT_LOG} || true", quiet=True)
    lines = out.strip().splitlines()
    return sw_name(*key), bool(lines) and lines[-1] == "1"

with futures.ThreadPoolExecutor(max_workers=16) as pool:
    bad = [n for n, ok in pool.map(deploy_agent, switches) if not ok]
assert not bad, f"agents failed to start on: {bad}"
print(f"{len(switches)} bmv2 agents running")

# ---- 4) heartbeat relay on the ctl_main NODE (survives controller kills) ----
def _v6(ip):
    return f"::ffff:{ip}" if "." in ip else ip
RELAY = f"""import socket
rx = socket.socket(socket.AF_INET6, socket.SOCK_DGRAM)
rx.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 0)
rx.bind(("", {RELAY_PORT}))
tx = socket.socket(socket.AF_INET6, socket.SOCK_DGRAM)
DSTS = (("::ffff:127.0.0.1", {HB_PORT}), ("{_v6(ctl_follow_ip)}", {HB_PORT}))
while True:
    data, _ = rx.recvfrom(4096)
    for dst in DSTS:
        try:
            tx.sendto(data, dst)
        except OSError:
            pass
"""
ctl_main.execute("cat > /tmp/hb_relay.py << 'PYEOF'\n" + RELAY + "PYEOF", quiet=True)
print("hb_relay.py staged on ctl_main")

# ---- 5) rough clock offsets (recovery times subtract cross-node clocks) ----
def clock_offset(node):
    t0 = time.time()
    out, _ = node.execute("date +%s.%N", quiet=True)
    t1 = time.time()
    return float(out.strip()) - (t0 + t1) / 2, (t1 - t0) / 2

print("clock offsets vs notebook (uncertainty = half the SSH round trip):")
for label, node in [("rcv0", receivers[0]), ("rcv7", receivers[7]),
                    ("sw2n1", switches[(2, 1)]), ("sw3n2", switches[(3, 2)])]:
    off, unc = clock_offset(node)
    print(f"  {label:6}: {off:+.3f}s (+/- {unc:.3f}s)")

# ---- 6) stop any previous workload ----
jobs = [n.execute_thread("sudo killall -q sender receiver controller 2>/dev/null; "
                         "sudo pkill -f '[c]ontroller_ha.py' 2>/dev/null; true")
        for n in end_hosts]
for j in futures.as_completed(jobs):
    j.result()
print("previous workload stopped -- setup complete")

In [ ]:
def route_from(entry_idx, recv_idx):
    """Switches a frame entering at rank-0 node entry_idx takes to recv_idx."""
    node = entry_idx
    path = [(0, node)]
    for s in range(K):
        if bit(node, s) != bit(recv_idx, s):
            node ^= (1 << s)
        path.append((s + 1, node))
    return path

killed_switches = set()

def kill_switch(key):
    """Fault: kill the BMv2 process (a full switch failure). Returns t_fail."""
    def f():
        out, _ = switches[key].execute("sudo killall simple_switch; date +%s.%N",
                                       quiet=True)
        killed_switches.add(key)
        return float(out.strip().splitlines()[-1])
    return f

def drop_entry(key, sid):
    """Fault: re-point one routing-MAC entry to drop (a path failure)."""
    mac = routing_mac[sid]
    handle = next(i for i, rl in enumerate(sw_rules[key])
                  if rl.get("route_mac", rl.get("dst_mac")) == mac)
    def f():
        out, _ = switches[key].execute(
            f"echo 'table_modify {P4_TABLE} drop {handle}' | simple_switch_CLI "
            f"> /dev/null 2>&1; date +%s.%N", quiet=True)
        return float(out.strip().splitlines()[-1])
    return f

def ctl_log_text(node, log):
    out, _ = node.execute(f"cat {RUN_DIR}/{log} 2>/dev/null", quiet=True)
    return sansi(out)

def ctl_events(text):
    """Unique event-log lines, in order (the TUI reprints its ring buffer)."""
    return list(dict.fromkeys(
        l.strip() for l in text.splitlines() if re.match(r"\s*\[\d\d:\d\d:\d\d\]", l)))

def parse_mapping(text):
    """sender id -> receiver idx from the LAST routing-table state in the log."""
    mapping = {}
    for m in re.finditer(r"(rcv\d+)\s*:\s*((?:snd\d+\s?)*)", text):
        for s in m.group(2).split():
            mapping[int(s[3:])] = int(m.group(1)[3:])
    return mapping

# fablib's shared execute_thread pool can be starved by long-lived
# foreground channels (each simple_switch relaunched by load_program holds
# one for the switch's lifetime), so experiment concurrency runs plain
# execute() calls on a private pool instead.
exp_pool = futures.ThreadPoolExecutor(max_workers=48)

def exec_bg(node, cmd):
    return exp_pool.submit(node.execute, cmd, quiet=True)

def start_exp_workload():
    # pkill separately: the launch command below contains the plain
    # script path, so a combined command would pkill its own shell
    ctl_main.execute("sudo pkill -f '[h]b_relay.py' 2>/dev/null; true", quiet=True)
    ctl_main.execute(f"mkdir -p {RUN_DIR}; "
                     f"sudo nohup python3 /tmp/hb_relay.py > {RUN_DIR}/relay.log 2>&1 &",
                     quiet=True)
    ctl_cmd = (f"sudo ./build/controller -m {EXP_MISS} -g {EXP_GRACE_MS} "
               f"-s switches.map topology.yaml {HB_PORT}")
    for node, role, peer, log in ((ctl_main, "main", ctl_follow_ip, "ctl_main.log"),
                                  (ctl_follow, "follower", ctl_main_ip, "ctl_follow.log")):
        node.execute(f"cd {REMOTE_DIR} && mkdir -p {RUN_DIR} && "
                     f"sudo nohup python3 ./artifact/controller_ha.py --role {role} "
                     f"--peer {peer} --hb-port {HA_PORT} -- {ctl_cmd} "
                     f"> {RUN_DIR}/{log} 2>&1 &", quiet=True)
    time.sleep(6)   # main controller: topology load + 32 agent PINGs
    jobs = [exec_bg(receivers[r],
                f"cd {REMOTE_DIR} && mkdir -p {RUN_DIR} && "
                f"sudo nohup ./build/receiver -i {EXP_HB_MS} {rx_dev[r]} {r} "
                f"{ctl_main_ip} {RELAY_PORT} > {RUN_DIR}/rcv{r}.log 2>&1 &")
            for r in range(NUM_RECEIVERS)]
    for j in futures.as_completed(jobs):
        j.result()
    time.sleep(2)
    jobs = [exec_bg(senders[sid],
                f"cd {REMOTE_DIR} && mkdir -p {RUN_DIR} && "
                f"sudo nohup timeout {EXP_SEND_SECS} ./build/sender -b -r {EXP_RATE} -l {SEND_LEN} "
                f"{snd_dev[sid]} {rx_mac[snd_target[sid]]} {sid} "
                f"> {RUN_DIR}/snd{sid}.log 2>&1 &")
            for sid in senders]
    for j in futures.as_completed(jobs):
        j.result()
    print("experiment workload running")

def stop_exp_workload(kill_relay=True):
    jobs = [exec_bg(n, "sudo killall -q sender receiver controller 2>/dev/null; "
                             "sudo pkill -f '[c]ontroller_ha.py' 2>/dev/null; true")
            for n in end_hosts]
    for j in futures.as_completed(jobs):
        j.result()
    if kill_relay:
        ctl_main.execute("sudo pkill -f '[h]b_relay.py' 2>/dev/null; true", quiet=True)

def run_fault_case(sid, fault_fn, ctl_node, ctl_log, label):
    mac = routing_mac[sid]
    cur = parse_mapping(ctl_log_text(ctl_node, ctl_log)).get(sid, pair_of_sender(sid))
    watchers = {r: exec_bg(receivers[r],
                    f"sudo timeout 90 tcpdump -i {rx_dev[r]} -tt -c 1 -nn "
                    f"'ether src {mac}' 2>/dev/null")
                for r in range(NUM_RECEIVERS) if r != cur}
    time.sleep(3)   # arm the captures
    pre_events = set(ctl_events(ctl_log_text(ctl_node, ctl_log)))

    t_fail = fault_fn()
    print(f"[{label}] snd{sid} (at rcv{cur}): fault injected, t_fail={t_fail:.6f}")

    t_appear = target = None
    deadline = time.time() + 90
    while time.time() < deadline and t_appear is None:
        for r, w in watchers.items():
            if w.done():
                out = (w.result()[0] or "").strip()
                if out:
                    t_appear, target = float(out.split()[0]), r
                    break
        time.sleep(0.5)
    for r in watchers:   # stop leftover captures
        if r != target:
            receivers[r].execute("sudo pkill -f '[t]cpdump -i' 2>/dev/null; true",
                                 quiet=True)

    events = [e for e in ctl_events(ctl_log_text(ctl_node, ctl_log))
              if e not in pre_events]
    print(f"[{label}] controller events:")
    for e in events:
        print(f"    {e}")
    if t_appear is None:
        print(f"[{label}] NO RECOVERY within 90s")
    else:
        print(f"[{label}] traffic reappeared at rcv{target} "
              f"({site_of(target)}), t_appear={t_appear:.6f}")
        print(f"[{label}] RECOVERY = {(t_appear - t_fail) * 1000:.1f} ms "
              f"(cross-node clocks, see setup offsets)")
    return {"sid": sid, "old": cur, "new": target, "t_fail": t_fail,
            "t_appear": t_appear, "events": events}

def wait_takeover(timeout=60):
    t0 = time.time()
    while time.time() - t0 < timeout:
        txt = ctl_log_text(ctl_follow, "ctl_follow.log")
        if "taking over" in txt:
            return time.time() - t0
        time.sleep(1)
    raise TimeoutError("follower did not take over")

def wait_follower_converged(quiet_secs=8, timeout=90):
    """A fresh follower re-derives moved senders via convergence failovers;
    wait until its event log is quiet before injecting the next fault."""
    last, last_change, t0 = None, time.time(), time.time()
    while time.time() - t0 < timeout:
        decs = [e for e in ctl_events(ctl_log_text(ctl_follow, "ctl_follow.log"))
                if "DECISION" in e]
        if decs != last:
            last, last_change = decs, time.time()
        elif time.time() - last_change > quiet_secs:
            return parse_mapping(ctl_log_text(ctl_follow, "ctl_follow.log"))
        time.sleep(1)
    raise TimeoutError("follower state did not settle")

def provenance_paths(sid, t0, t1, t_fail):
    """Hop chains of snd<sid>'s flow (ip.src filter) before vs after t_fail,
    merged across both slices' analyzers by packet UID."""
    merged = {}
    for letter in ("A", "B"):
        try:
            d = slice_handles[letter]["slice"].dump_provenance(
                name=f"prov{letter}{sid}", filterin=f"src 10.0.{sid}.1",
                tstart=str(int(t0)), tend=str(int(t1) + 1), tformat="epoch")
        except Exception as e:
            print(f"  provenance query slice {letter} failed: {e}")
            continue
        for pkt, hops in d.items():
            merged.setdefault(pkt, []).extend(hops)
    before, after = {}, {}
    for pkt, hops in merged.items():
        hops.sort(key=lambda h: float(h.get("epoch", h.get("time", 0))))
        chain = " -> ".join([hops[0]["tx_host"]] + [h["rx_host"] for h in hops])
        bucket = before if float(hops[0].get("epoch", 0)) < t_fail else after
        bucket[chain] = bucket.get(chain, 0) + 1
    for label, chains in (("BEFORE fault", before), ("AFTER fault", after)):
        print(f"  {label}: {sum(chains.values())} packets, {len(chains)} distinct paths")
        for chain, cnt in sorted(chains.items(), key=lambda kv: -kv[1]):
            print(f"    {cnt:5}x  {chain}")
    return before, after

def reset_experiment(restart_workload=True):
    stop_exp_workload()
    for key in sorted(killed_switches):
        print(f"restarting {sw_name(*key)}")
        switches[key].start_switch(force=True)
    killed_switches.clear()
    def reprogram(key):
        sw = switches[key]
        if not sw.load_program(P4_LOCAL):
            return sw_name(*key), "load_program FAILED"
        n = sum(bool(sw.run_command(
                    f"table_add {P4_TABLE} forward "
                    f"{mac_to_hex(r['route_mac'])} => {r['port']}"))
                for r in sw_rules[key])
        return sw_name(*key), f"{n}/{len(sw_rules[key])} rules"
    with futures.ThreadPoolExecutor(max_workers=16) as pool:
        for name, msg in pool.map(reprogram, switches):
            if "FAILED" in msg or "/" not in msg:
                print(f"  {name}: {msg}")
    # agents cache entry handles; a reprogram invalidates them
    def restart_agent(key):
        sw = switches[key]
        sw.execute("sudo pkill -f '[b]mv2_switch_agent.py' 2>/dev/null; true",
                   quiet=True)
        sw.execute(
            f"rm -f /tmp/bmv2_switch_agent_stop; "
            f"sudo nohup python3 -u /tmp/bmv2_switch_agent.py {AGENT_CONF} > {AGENT_LOG} 2>&1 &",
            quiet=True)
    with futures.ThreadPoolExecutor(max_workers=16) as pool:
        list(pool.map(restart_agent, switches))
    for letter in ("A", "B"):
        slice_handles[letter]["slice"].reset_analyzer()
    print("dataplane reset: switches reprogrammed, agents + analyzers restarted")
    if restart_workload:
        start_exp_workload()

print("experiment helpers defined")

## 13. Failover experiment A: intra-site

Fault model: a **switch death** (`killall simple_switch` on the sender's rank-2
switch). That switch carries both the current path and the cross-site sibling
path, so the controller's first failover (to the sibling, per DFS order) stays
dark; after the grace period a second failover lands on an intra-site receiver.
Expected for snd2 (rcv1, CERN): rcv5 (dark) -> **rcv3 (CERN)**. Then the main
controller is killed (follower takes over through the relay) and a second switch
death is timed under the follower: snd1 (rcv0): rcv4 (dark) -> **rcv2 (CERN)**.

In [ ]:
# Fresh dataplane + workload, then confirm steady state.
reset_experiment()
time.sleep(EXP_MISS * EXP_HB_MS / 1000 + 10)
print(ctl_log_text(ctl_main, "ctl_main.log")[-1200:])

In [ ]:
INTRA1_SID = 2
key = route_from(pair_of_sender(INTRA1_SID), snd_target[INTRA1_SID])[2]
print(f"fault: killing {sw_name(*key)} (rank-2 switch of snd{INTRA1_SID}'s path)")
res_intra1 = run_fault_case(INTRA1_SID, kill_switch(key),
                            ctl_main, "ctl_main.log", "intra-1")

print("\nprovenance (before vs after the fault):")
provenance_paths(INTRA1_SID, res_intra1["t_fail"] - 20,
                 (res_intra1["t_appear"] or time.time()) + 10,
                 res_intra1["t_fail"])

In [ ]:
# Kill the main controller PROCESS (the relay on its node survives).
ctl_main.execute("sudo pkill -f '[c]ontroller_ha.py'; sudo killall -q controller; true",
                 quiet=True)
t_takeover = wait_takeover()
print(f"follower took over {t_takeover:.1f}s after the kill")
mapping = wait_follower_converged()
print(f"follower converged; mapping: {mapping}")

INTRA2_SID = 1
key = route_from(pair_of_sender(INTRA2_SID), snd_target[INTRA2_SID])[2]
print(f"fault: killing {sw_name(*key)} (rank-2 switch of snd{INTRA2_SID}'s path)")
res_intra2 = run_fault_case(INTRA2_SID, kill_switch(key),
                            ctl_follow, "ctl_follow.log", "intra-2")

print("\nprovenance (before vs after the fault):")
provenance_paths(INTRA2_SID, res_intra2["t_fail"] - 20,
                 (res_intra2["t_appear"] or time.time()) + 10,
                 res_intra2["t_fail"])

## 14. Failover experiment B: cross-site

Fault model: a **path failure** (the sender's routing-MAC entry at its rank-3
switch re-pointed to `drop`). The first alternative path is the other site's
sibling receiver, so a single fault forces the failover **across the VXLAN
cross-site links**. Expected: snd3 (rcv2, CERN) -> **rcv6 (PRIN)**; after the
controller takeover, snd4 (rcv3, CERN) -> **rcv7 (PRIN)**. The after-fault
provenance chains should traverse the tunnel-node interfaces.

In [ ]:
# Fresh dataplane + workload (also restores section 13's killed switches
# and re-launches the main controller pair).
reset_experiment()
time.sleep(EXP_MISS * EXP_HB_MS / 1000 + 10)
print(ctl_log_text(ctl_main, "ctl_main.log")[-1200:])

In [ ]:
CROSS1_SID = 3
key = route_from(pair_of_sender(CROSS1_SID), snd_target[CROSS1_SID])[K]
print(f"fault: dropping snd{CROSS1_SID}'s entry at {sw_name(*key)} (rank-3)")
res_cross1 = run_fault_case(CROSS1_SID, drop_entry(key, CROSS1_SID),
                            ctl_main, "ctl_main.log", "cross-1")

print("\nprovenance (before vs after the fault):")
provenance_paths(CROSS1_SID, res_cross1["t_fail"] - 20,
                 (res_cross1["t_appear"] or time.time()) + 10,
                 res_cross1["t_fail"])

In [ ]:
ctl_main.execute("sudo pkill -f '[c]ontroller_ha.py'; sudo killall -q controller; true",
                 quiet=True)
t_takeover = wait_takeover()
print(f"follower took over {t_takeover:.1f}s after the kill")
mapping = wait_follower_converged()
print(f"follower converged; mapping: {mapping}")

CROSS2_SID = 4
key = route_from(pair_of_sender(CROSS2_SID), snd_target[CROSS2_SID])[K]
print(f"fault: dropping snd{CROSS2_SID}'s entry at {sw_name(*key)} (rank-3)")
res_cross2 = run_fault_case(CROSS2_SID, drop_entry(key, CROSS2_SID),
                            ctl_follow, "ctl_follow.log", "cross-2")

print("\nprovenance (before vs after the fault):")
provenance_paths(CROSS2_SID, res_cross2["t_fail"] - 20,
                 (res_cross2["t_appear"] or time.time()) + 10,
                 res_cross2["t_fail"])

In [ ]:
# Restore the plain dataplane (switches reprogrammed, agents fresh) and stop
# the experiment workload.
reset_experiment(restart_workload=False)
print("experiment state cleaned up")

## 15. Teardown

Stop the switches (both slices) and (optionally) delete both slices.

In [ ]:
# Stop BMv2 on every switch (merged across both slices).
for key, sw in switches.items():
    try:
        sw.stop_switch()
    except Exception as e:
        print(f"{sw_name(*key)}: {e}")

# Uncomment to delete both slices:
# slice_handles["A"]["slice"].delete()
# slice_handles["B"]["slice"].delete()
# print("both slices deleted")